# Foursquare Open Source Placesを使ってみる

## 1. 今回やること
DuckDBからFoursquare Open Source Placesへ接続し、吉祥寺周辺のPOIを取得して、カフェを抽出します。

今回は取得方法の確認が目的です。POI（店舗・施設などの地点情報）の網羅性評価は行いません。
詳細なライセンス・実行手順・内部仕様は[利用ガイド](../docs/foursquare.md)、実測結果は[検証記録](../docs/foursquare-validation.md)を参照してください。

## 2. 準備
必要なのはPython / uv環境、DuckDB、[Places Portal](https://places.foursquare.com/)のaccess tokenです。
リポジトリのルートで `uv sync --locked --group notebooks` を実行すると、DuckDBとNotebook用の環境も揃います。

ルートの `.env` に `FSQ_OS_PLACES_TOKEN` を設定してください。**tokenはNotebookへ直接書きません。**
このNotebookはリポジトリのルートまたは `notebooks/` から、上から順に実行します。

In [ ]:
from geoai_open_lab.foursquare import (
    connect_foursquare, load_aoi, fetch_places,
    filter_places_by_category, save_results,
)

## 3. Foursquareへ接続
Portalのtokenを使い、DuckDBからIcebergカタログへ接続します。
関数内で必要な拡張を読み込み、データのsnapshot（特定時点の版）を固定します。

In [ ]:
con = connect_foursquare()
print("Foursquareに接続できました。")

## 4. 吉祥寺周辺のPOIを取得
共通のAOI設定（対象地域の矩形）を読み込み、日本を表す `JP` と組み合わせます。
取得するのはID順の最大500件です。まず名前と位置を見てみましょう。

In [ ]:
kichijoji = load_aoi("configs/aoi/kichijoji.toml")
places = fetch_places(con, bbox=kichijoji, country="JP", limit=500)
places[["name", "latitude", "longitude", "country"]].head()

## 5. Categoriesからカフェを抽出
OS Categoriesの「Café」「Coffee Shop」とその子カテゴリを、POIのカテゴリIDに照合します。
`Cafe` は `Café` にも対応します。店名の検索ではなく、カテゴリ情報による抽出です。
CafeteriaやInternet Cafeを部分一致で含めません。

In [ ]:
cafes = filter_places_by_category(con, places, categories=["Cafe", "Coffee Shop"])
cafes[["name", "latitude", "longitude"]].head(10)

## 6. 結果と注意点
表示する件数は、今回取得したサンプルの中の件数です。

In [ ]:
print(f"取得したPOIサンプル: {len(places)}件")
print(f"サンプル内のカフェ・コーヒーショップ: {len(cafes)}件")

2026-09-24の検証では、日本のPOIを取得し、OS CategoriesとのJOINでカフェ系POIを抽出できました。
取得したサンプルは **POI 500件、その中のカフェ・コーヒーショップ37件** でした。

- 今回は最大500件の動作確認用サンプルです。検証時は上限に達しており、吉祥寺周辺全体のPOI件数ではありません。
- カフェ件数もサンプル内の値で、地域全体の店舗数ではありません。閉店記録を含みます。
- データの版や条件を変えると結果も変わります。網羅性・正確性・他データとの比較は後続記事で扱います。

最後に、取得結果と再現用の記録をローカルのGit管理外 `data/` に保存し、接続を閉じます。

In [ ]:
result_dir = save_results(con)
con.close()